This notebook illustrates a model-in-the-loop path from a constrained QP formulation to an FPGA-ready hardware accelerator. CVXPyGen is used to expose the solver structure from the generated PDA-QP algorithm, while Chisel is used to realize that structure as synthesizable hardware and verification code. The workflow deliberately decouples three questions in the end-to-end design workflow:

- **why** the optimization problem exists;
- **what** numerical operations the solver must perform, and
- **how** those operations should be synthesized, compiled, and regression-tested in hardware.

By keeping those concerns in separate DSLs, the flow preserves mathematical intent at the model level while enabling systematic hardware generation and verification downstream.

In [1]:
from pathlib import Path

from cvxpygen.hwgen.codegen import generateHWAccCode, generateTestbenchCode
from cvxpygen.hwgen.decode import decode
from cvxpygen.hwgen.firmware_models import LINUX_AMD64
from cvxpygen.hwgen.implement import quantize, specialize
from cvxpygen.hwgen.models import DataFormat, MultiplyAddFix16

## (Skipped) Problem formulation: Why the optimization exists

(Skipped. To be included once the `cvxpygen -> cvxpygen.hwgen` pipeline is complete.)

## The algorithm parameters: What must be computed

At this stage, CVXPyGen together with the PDA-QP code generation plugin produces two artifacts: pdaqp.c and pdaqp.h. These files encode the solver parameters: binary tree data structures, halfspaces parameters, and iteration logic.

We then parse these C files into `cvxpygen.hwgen` intermediate representations (CVXPy-HWGen IR). Next, we download the FPGA design project from Github. In parallel, the extracted algorithm parameters in the IR are exported into the hardware design as a configuration file `Constants.scala`.

The critical transformation here is **quantization**. Solver parameters are converted from 32-bit floating-point representations into fixed-point decimal format suitable for baremetal (FPGA) hardware.

Notably, integer overflow is not treated/handled as a hardware exception to be guarded against in the FPGA. Instead, it is **eliminated by construction**. We expect the constrained QP formulation (in CVXPy) already bounds the valid range of all variables. By propagating these bounds into fixed-point design, we ensure that all arithmetic remains within safe limits.

This illustrates a key advantage of **algorithm-hardware co-design**:

> Numerial constraints at the algorithm level simplify hardware complexity.

As a result, the design avoids costly features such as:

- floating point arithmetic logic units (ALUs), and
- integer underflow/overflow logic,

..., replacing them with bit-exact, resource-efficient fixed-point arithmetic circuits (in FPGA) derived **from first principles**.

In [2]:
prefix = Path("../test-vectors/pid")
algorithm_config = decode(prefix / "pdaqp.c", prefix / "pdaqp.h")

hw_config = specialize(algorithm_config)
hw_config_fix16 = quantize(hw_config, DataFormat.FIXQ2_14)
assert isinstance(hw_config_fix16.feedback_module.param, MultiplyAddFix16)
assert isinstance(hw_config_fix16.half_planes_module.param, MultiplyAddFix16)

In [3]:
!git clone https://github.com/antonysigma/pdaqp-solver-chisel.git

fatal: destination path 'pdaqp-solver-chisel' already exists and is not an empty directory.


In [4]:
pdaqp_solver_chisel_path = Path("./pdaqp-solver-chisel/")
outfile = generateHWAccCode(
    hw_config_fix16, pdaqp_solver_chisel_path / "pdaqp_design/src"
)
assert outfile.exists()

## Automatic design handover to build system: How it is synthesized

The solver configuration is then **lowered into the hardware design domain** using Chisel HDL.

Chisel serves as a domain-specific language (DSL), responsible for:

- generating parameterized and reusable hardware modules;
- connecting modoubles with databuses;
- defining pipeline structures and control flow; and
- enforcing bit-accurate integer and state representations.

At this point, the workflow transitions from algorithm description to computer-aided design (CAD). The FPGA project's build system (Meson DSL and Mill DSL) automatically configures the Chisel environment, downloading external dependencies if necessary.

In particular, the build system identifies the need for a **reference software implementation** of the PDA-QP solver. It then fetches the corresponding C++ sources from Github, ensuring that the hardware design can be validated against a known-correct baseline, commonly known as the "Golden copy" in the hardware design world.

To support model-in-the-loop design verification, the project also retrieves the test frameworks (Catch2 and LLVM-fuzzer) on the fly.

This stage mirrors the earlier separation between algorithm and hardware:

- Chisel encapsulates *how* the computation is realized in hardware; while
- Meson and Mill build system encapsulates *how* the project is constructed, configured, and validated.

Meson and Mill themselves functions as a 3rd DSL in the workflow, focused on **build orchestration and dependency management**. By isolating the concerns, the flow avoid entangling algorithm logic, design logic, with tooling infrastructure.

In [16]:
!meson setup --buildtype=release \
  -Db_lto=true \
  --native-file pdaqp-solver-chisel/native.ini \
  pdaqp-solver-chisel/ \
  pdaqp-solver-chisel/build/ \
  --wipe

The Meson build system
Version: 1.2.3
Source dir: /home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel
Build dir: /home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel/build
Build type: native build
Project name: pdaqp_design
Project version: undefined
C++ compiler for the host machine: ccache /usr/bin/clang++-18 -stdlib=libstdc++ (clang 18.1.8 "Ubuntu clang version 18.1.8 (11~20.04.2)")
C++ linker for the host machine: /usr/bin/clang++-18 -stdlib=libstdc++ ld.lld 18.1.8
Host machine cpu family: x86_64
Host machine cpu: x86_64
Program verilator found: YES (/usr/bin/verilator)
Message: Detected Verilator version: 4.028
Run-time dependency threads found: YES
Program run_at_src_root.sh found: YES (/home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel/scripts/run_at_src_root.sh)
Program run_at_current_build_dir.sh found: YES (/home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel/scripts/run_at_current_build_dir.sh)

Executing subproject mill-build-system 

mill-bui

## Model-in-the-loop testing

Returning to the CVXPyGen Python environment, the solver design is once again **lowered**, except this time into a firmware-level configuration suitable for the C++ reference implementation (known as the "golden copy" by the FPGA design world). The resulting artifacts are exported as C++17 header files: `problem-scale.hpp` and `problem-definition.hpp`.

These generated definitions are then integrated into the PDA-QP C++ solver project via the Meson build system. By incorporating the exact same problem parameters (i.e. quantized to 16-bit fixed-point decimals) used in the hardware generation, the C++ solver becomes the **reference model** for validation.

This enables the model-in-the-loop design workflow:

- the C++ solver executes the problem using identical maths;
- the hardware implementation (converted to SystemVerilog by Chisel, followed by C++ emulation by Verilator) is exercised under the same conditions;
- results are compared to ensure numerical and functional equivalence.

To improve the design cadence, both the reference design (in C++17) and the device under test (DUT, emulated by Verilator framework) are accelerated on the execution environment (Linux 64-bit OS with SIMD vectorized instructions and Multi-core CPUs).

In [8]:
fixed_point_precision = 14
constants_hpp, problem_def_hpp = generateTestbenchCode(
    algorithm_config,
    fixed_point_precision,
    LINUX_AMD64,
    pdaqp_solver_chisel_path / "subprojects/pdaqp-solver-cpp/examples/user",
)
assert constants_hpp.exists()
assert problem_def_hpp.exists()

In [18]:
!meson configure -Dpdaqp-solver-cpp:name=user ./pdaqp-solver-chisel/build

In [20]:
!ninja -C ./pdaqp-solver-chisel/build all

ninja: Entering directory `./pdaqp-solver-chisel/build'
[5/208] Compiling C++ object libverila...erilator_include_verilated_fst_c.cpp.oK
In file included from /usr/share/verilator/include/verilated_fst_c.cpp:37:
/usr/share/verilator/include/gtkwave/fstapi.c:5984:14: warning: variable 'secnum' set but not used [-Wunused-but-set-variable]
 5984 | unsigned int secnum = 0;
      |              ^
1 warning generated.
[187/208] Generating pdaqp_design/mock... command (wrapped by meson to set env)K
Current working directory: /home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel
/home/antony/Projects/cvxpygen/examples/pdaqp-so...aqp-solver-chisel/build/pdaqp_design/mock-tests
79] pdaqp_design.runMain
79/79, SUCCESS] /home/antony/Projects/cvxpygen/example...r-chisel/build/pdaqp_design/mock-tests 3s
[192/208] Generating pdaqp_design/src/... command (wrapped by meson to set env)
Current working directory: /home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel
Another Mill process is ru

## Automated report generation

As illustrated below, the first 3 tests are executed in parallel across multiple CPU cores, each test further benefiting from SIMD acceleration within a single thread. In addition, the Meson build system performs self-checks on the Catch2 test framework iteself, ensuring the integrity of the testing infrastructure end-to-end.

The structure of the binary decision tree, the core component of the PDA-QP solver, is validated by walking all the branches of the (FPGA) hardware circuit, and compare the leaf node IDs against that of the C++ reference implementation.

The more time-intensive tests, i.e. `DotProduct` and `GEMV`, are driven by Monte-Carlo-style validation using randomized solver inputs. Over approximately 9 seconds, on the order of 10 million (`10^7`) parameter sets are generated as 16-bit fixed-point decimal values, and fed into both the C++ reference solver and the FPGA hardware model.

The outputs are then compared for **bit-exact equivalence**, systemtically exposing edge conditions and corner cases that would be difficult to capture with deterministic tests alone.



In [21]:
!ninja -C ./pdaqp-solver-chisel/build test

ninja: Entering directory `./pdaqp-solver-chisel/build'
[0/1] Running all tests.
1/6 pdaqp_design / Hello world                                      OK              0.02s
2/6 pdaqp_design:gemv / Feedback offset matches                     OK              0.01s   9 subtests passed
3/6 pdaqp_design:tree-walk / Feedback offset matches                OK              0.01s   34 subtests passed
4/6 catch2 / SelfTest                                               OK              0.32s
5/6 pdaqp_design / Dot product and less-than-and-equal              OK              9.01s
6/6 pdaqp_design:gemv / Matrix-vector multiplication and add        OK              8.69s

Ok:                 6   
Expected Fail:      0   
Fail:               0   
Unexpected Pass:    0   
Skipped:            0   
Timeout:            0   

Full log written to /home/antony/Projects/cvxpygen/examples/pdaqp-solver-chisel/build/meson-logs/testlog.txt


## (TODO) FPGA utilization report

In [22]:
!ninja -C ./pdaqp-solver-chisel/build yosys

ninja: Entering directory `./pdaqp-solver-chisel/build'
ninja: error: unknown target 'yosys'
